In [2]:
import pandas as pd
import numpy as np

In [5]:
reload_results =  pd.read_csv(r"G:\Shared drives\Kerlin lab shared\Manuscripts\Reload Paper\Figures\Supplemental\supplement_psf\supplement_500kHz_spots\081925_500KHz_spot_data\spots2_9\results.csv")

factor = 2 * np.sqrt(2 * np.log(2)) 

reload_results["FWHM"] = factor * reload_results["h_sigma"]
reload_results["FWHM_err"] = factor * reload_results["h_sigma_err"]

reload_results_sorted = reload_results.sort_values(by="cx_corrected", ascending=True)

reload_results_sorted.to_csv(r"G:\Shared drives\Kerlin lab shared\Manuscripts\Reload Paper\Figures\Supplemental\supplement_psf\supplement_500kHz_spots\081925_500KHz_spot_data\spots2_9\spotdata_workup\results_sorted.csv", index=False)


In [6]:
reload_results_sorted = pd.read_csv(r"G:\Shared drives\Kerlin lab shared\Manuscripts\Reload Paper\Figures\Supplemental\supplement_psf\supplement_500kHz_spots\081925_500KHz_spot_data\spots2_9\spotdata_workup\results_sorted.csv")

# Initialize new columns
reload_results_sorted["ssdistance"] = 0.0
reload_results_sorted["ssdistance_err"] = 0.0
reload_results_sorted["avg_FWHM"] = 0.0
reload_results_sorted["avg_FWHM_err"] = 0.0
reload_results_sorted["spots_between"] = 0.0
reload_results_sorted["spots_between_err"] = 0.0

for i in range(1, len(reload_results_sorted)):
    dx = reload_results_sorted.loc[i, "cx_corrected"] - reload_results_sorted.loc[i-1, "cx_corrected"]
    dy = reload_results_sorted.loc[i, "cy_corrected"] - reload_results_sorted.loc[i-1, "cy_corrected"]
    d = np.sqrt(dx**2 + dy**2)
    
    dx_err = np.sqrt(reload_results_sorted.loc[i, "cx_corrected_err"]**2 + reload_results_sorted.loc[i-1, "cx_corrected_err"]**2)
    dy_err = np.sqrt(reload_results_sorted.loc[i, "cy_corrected_err"]**2 + reload_results_sorted.loc[i-1, "cy_corrected_err"]**2)
    if d != 0:
        d_err = np.sqrt(((dx/d)**2) * dx_err**2 + ((dy/d)**2) * dy_err**2)
    else:
        d_err = 0.0
    
    avg_fwhm = 0.5 * (reload_results_sorted.loc[i, "FWHM"] + reload_results_sorted.loc[i-1, "FWHM"])
    avg_fwhm_err = 0.5 * np.sqrt(reload_results_sorted.loc[i, "FWHM_err"]**2 + reload_results_sorted.loc[i-1, "FWHM_err"]**2)
    
    # Spots between + error propagation
    if avg_fwhm != 0:
        spots = (d / avg_fwhm) - 1
        spots_err = np.sqrt((d_err / avg_fwhm)**2 + ((-d / avg_fwhm**2) * avg_fwhm_err)**2)
    else:
        spots, spots_err = 0.0, 0.0
    
    # Assign
    reload_results_sorted.loc[i, "ssdistance"] = d
    reload_results_sorted.loc[i, "ssdistance_err"] = d_err
    reload_results_sorted.loc[i, "avg_FWHM"] = avg_fwhm
    reload_results_sorted.loc[i, "avg_FWHM_err"] = avg_fwhm_err
    reload_results_sorted.loc[i, "spots_between"] = spots
    reload_results_sorted.loc[i, "spots_between_err"] = spots_err

# Save updated CSV
reload_results_sorted.to_csv(r"G:\Shared drives\Kerlin lab shared\Manuscripts\Reload Paper\Figures\Supplemental\supplement_psf\supplement_500kHz_spots\081925_500KHz_spot_data\spots2_9\spotdata_workup\reload_results_sorted_wspotcalcs.csv", index=False)


In [8]:
# Load the CSV with spot calculations
results = pd.read_csv(r"G:\Shared drives\Kerlin lab shared\Manuscripts\Reload Paper\Figures\Supplemental\supplement_psf\supplement_500kHz_spots\081925_500KHz_spot_data\spots2_9\spotdata_workup\reload_results_sorted_wspotcalcs.csv")

# Number of measured spots (rows)
num_spots = len(results)

# Sum of spots_between
spots_sum = results["spots_between"].sum()

# Total spots
total_spots = spots_sum + num_spots

# Total error (RSS of individual errors)
total_spots_err = np.sqrt(np.sum(results["spots_between_err"]**2))

print("Number of spots (rows):", num_spots)
print("Sum of spots_between:", spots_sum)
print("Total spots:", total_spots)
print("Total spots error:", total_spots_err)


Number of spots (rows): 9
Sum of spots_between: 41.93829570951683
Total spots: 50.93829570951683
Total spots error: 0.22424772222463607
